# Customer Segmentation & RFM Analysis

This notebook provides a reproducible, top-to-bottom workflow for loading the Online Retail dataset, building RFM metrics in PostgreSQL, assigning customer segments, validating the results, and exporting the Power BI input file.

## Prerequisites

1. Download **Online Retail.xlsx** from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/352/online+retail).
2. Put the file in the same directory as this notebook, or set the `RFM_DATA_PATH` environment variable to its full path.
3. Create an empty PostgreSQL database (default name: `project5_rfm`).
4. Optionally set `RFM_DB_HOST`, `RFM_DB_PORT`, `RFM_DB_NAME`, and `RFM_DB_USER`. The notebook prompts securely for the password.

Run all cells in order from a fresh kernel.


## 1. Imports and configuration


In [ ]:
import getpass
import io
import os
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg

DATA_PATH = Path(os.getenv("RFM_DATA_PATH", "Online Retail.xlsx")).expanduser()
DB_CONFIG = {
    "host": os.getenv("RFM_DB_HOST", "localhost"),
    "port": int(os.getenv("RFM_DB_PORT", "5432")),
    "dbname": os.getenv("RFM_DB_NAME", "project5_rfm"),
    "user": os.getenv("RFM_DB_USER", "postgres"),
}

print("pandas:", pd.__version__)
print("psycopg:", psycopg.__version__)
print("Dataset:", DATA_PATH.resolve())


## 2. Load and inspect the source data


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}\n"
        "Download Online Retail.xlsx from UCI or set RFM_DATA_PATH."
    )

df = pd.read_excel(DATA_PATH)

required_columns = {
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country"
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

print("Rows and columns:", df.shape)
print("First transaction:", df["InvoiceDate"].min())
print("Last transaction:", df["InvoiceDate"].max())
print("Distinct non-null customers:", df["CustomerID"].nunique())
print("Missing customer IDs:", int(df["CustomerID"].isna().sum()))
print("Negative quantities:", int((df["Quantity"] < 0).sum()))
print("Zero or negative prices:", int((df["UnitPrice"] <= 0).sum()))

df.head()


## 3. Connect to PostgreSQL


In [ ]:
db_password = getpass.getpass("PostgreSQL password: ")

conn = psycopg.connect(
    **DB_CONFIG,
    password=db_password,
    connect_timeout=10,
)

print("Connected to:", conn.info.dbname)


## 4. Load the raw transaction table

The table is truncated before loading, so rerunning the notebook does not duplicate transactions.


In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS retail_transactions_raw (
            invoice_no TEXT,
            stock_code TEXT,
            description TEXT,
            quantity INTEGER,
            invoice_date TIMESTAMP,
            unit_price NUMERIC,
            customer_id NUMERIC,
            country TEXT
        );
        TRUNCATE TABLE retail_transactions_raw;
    """)

buffer = io.StringIO()
df.to_csv(buffer, index=False, header=False, na_rep="\\N")
buffer.seek(0)

with conn.cursor() as cur:
    with cur.copy(r"""
        COPY retail_transactions_raw
        FROM STDIN
        WITH (FORMAT CSV, NULL '\N')
    """) as copy:
        copy.write(buffer.getvalue())

conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT customer_id) AS unique_customers,
            COUNT(*) FILTER (WHERE customer_id IS NULL) AS missing_customer_ids
        FROM retail_transactions_raw;
    """)
    print(cur.fetchone())


## 5. Clean transactions and build customer RFM metrics


In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        DROP TABLE IF EXISTS retail_transactions_clean;
        CREATE TABLE retail_transactions_clean AS
        SELECT
            invoice_no,
            stock_code,
            description,
            quantity,
            invoice_date,
            unit_price,
            customer_id,
            country,
            quantity * unit_price AS line_total
        FROM retail_transactions_raw
        WHERE customer_id IS NOT NULL
          AND quantity > 0
          AND unit_price > 0
          AND invoice_no NOT LIKE 'C%';

        DROP TABLE IF EXISTS customer_rfm;
        CREATE TABLE customer_rfm AS
        WITH reference_date AS (
            SELECT MAX(invoice_date)::date + 1 AS analysis_date
            FROM retail_transactions_clean
        )
        SELECT
            t.customer_id,
            (r.analysis_date - MAX(t.invoice_date)::date)::integer AS recency,
            COUNT(DISTINCT t.invoice_no)::integer AS frequency,
            ROUND(SUM(t.line_total), 2) AS monetary
        FROM retail_transactions_clean AS t
        CROSS JOIN reference_date AS r
        GROUP BY t.customer_id, r.analysis_date;
    """)
    conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT COUNT(*), MIN(recency), MAX(recency),
               MIN(frequency), MAX(frequency),
               ROUND(SUM(monetary), 2)
        FROM customer_rfm;
    """)
    print(cur.fetchone())


## 6. Score and segment customers


In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        DROP TABLE IF EXISTS customer_segments;

        CREATE TABLE customer_segments AS
        WITH scored AS (
            SELECT
                customer_id,
                recency,
                frequency,
                monetary,
                NTILE(5) OVER (ORDER BY recency DESC) AS r_score,
                NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
                NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
            FROM customer_rfm
        )
        SELECT
            *,
            CASE
                WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'VIP'
                WHEN r_score >= 3 AND f_score >= 3 THEN 'Loyal'
                WHEN r_score <= 2 AND f_score >= 3 THEN 'At Risk'
                WHEN r_score = 1 AND f_score <= 2 THEN 'Lost'
                ELSE 'Other'
            END AS segment
        FROM scored;
    """)
    conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT segment, COUNT(*) AS customers, ROUND(SUM(monetary), 2) AS revenue
        FROM customer_segments
        GROUP BY segment
        ORDER BY revenue DESC;
    """)
    for row in cur.fetchall():
        print(row)


## 7. Validate segment assignments and summarize results


In [ ]:
segments = pd.read_sql_query(
    "SELECT * FROM customer_segments ORDER BY customer_id",
    conn,
)

expected = np.select(
    [
        (segments.r_score >= 4) & (segments.f_score >= 4) & (segments.m_score >= 4),
        (segments.r_score >= 3) & (segments.f_score >= 3),
        (segments.r_score <= 2) & (segments.f_score >= 3),
        (segments.r_score == 1) & (segments.f_score <= 2),
    ],
    ["VIP", "Loyal", "At Risk", "Lost"],
    default="Other",
)

mismatches = segments.loc[segments.segment != expected]

print("Customers checked:", len(segments))
print("Distinct customer IDs:", segments["customer_id"].nunique())
print("Missing scores:", int(segments[["r_score", "f_score", "m_score"]].isna().sum().sum()))
print("Incorrect assignments:", len(mismatches))
print("QA result:", "PASS" if len(mismatches) == 0 else "FAIL")


In [ ]:
summary = segments.groupby("segment").agg(
    customers=("customer_id", "count"),
    revenue=("monetary", "sum"),
    avg_recency_days=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_spending=("monetary", "mean"),
)

summary["customer_pct"] = summary["customers"] / len(segments) * 100
summary["revenue_pct"] = summary["revenue"] / segments["monetary"].sum() * 100

print("Total customers:", len(segments))
print("Total monetary value:", round(segments["monetary"].sum(), 2))
print(summary.round(2).to_string())


## 8. Export the Power BI input


In [ ]:
OUTPUT_PATH = Path("customer_segments.csv")
segments.to_csv(OUTPUT_PATH, index=False)
print(f"Exported {len(segments):,} customers to {OUTPUT_PATH.resolve()}")


## 9. Close the database connection


In [ ]:
conn.close()
print("PostgreSQL connection closed.")
